# Canonical Correlation Analysis of 16S V1-V3 and V4 and metabolomics table with top VIPs

Date created: 1/29/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen

In [39]:
import biom
from biom import load_table
from biom.util import biom_open
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from skbio.stats.composition import clr
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

In [40]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [41]:
def save_as_biom(df: pd.DataFrame, output_path: str):
    """
    Save a pandas DataFrame as a BIOM table.

    Parameters:
    df (pd.DataFrame): The DataFrame to save.
    output_path (str): Path to the output BIOM file.
    """
    table = biom.table.Table(df.values, observation_ids=df.index, sample_ids=df.columns)
    with biom_open(output_path, 'w') as f:
        table.to_hdf5(f, "save biom")

In [42]:
# Read in the metadata
metadata = pd.read_csv('../Metadata/metadata_final_18062024.tsv', sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,0,0,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,36,72,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,26,69,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,0,0,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,13,90,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,19,77,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,0,0,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,0,0,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,31,55,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L


In [43]:
skin_group = "Acne_L"

In [44]:
# Read in metabolomics table with top VIPs
metaB_tbl = pd.read_csv('../Data/metabolomics/Run3_10252024/output/metaB_top-VIPs_final.csv')
metaB_tbl = metaB_tbl.set_index('SampleID')

# Read in 16S V1-V3 table
V1V3_biom_path = '../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom'
V1V3_tbl = biom_to_df(V1V3_biom_path)

# Read in 16S V4 table
V4_biom_path = '../Data/16S/Tables/16S_V4_Genus_collapsed_absolute.biom'
V4_tbl = biom_to_df(V4_biom_path)

In [45]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_V1V3_tbl = metaB_tbl[metaB_tbl.index.isin(V1V3_tbl.index)]

# Remove the 'group' column (only for VIP table)
# metaB_V1V3_tbl = metaB_V1V3_tbl.drop(columns=['group'])
metaB_V1V3_tbl.index.name = None
metaB_V1V3_tbl

,Urocanic acid,Isoleucine,Phenylalanine,Tryptophan,N-acyl lipid glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,39859.7580,2025607.40,970487.500,567510.060,0.0000,0.000,611709.700
LAMI.RD304.D11.C1,36750.4000,1434033.20,595568.940,328151.120,5375.8525,99470.690,78491.530
LAMI.RD304.D11.C2,0.0000,706197.75,303845.300,152392.720,2691.0256,157503.920,261996.800
LAMI.RD304.D0.C1,3345.8184,778566.10,293386.340,137201.390,3431.0034,0.000,427202.060
LAMI.RD304.D0.C2,24984.3220,2486485.20,1051277.400,459814.620,13703.5760,62253.426,223301.700
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,1250164.60,579058.800,498893.780,0.0000,0.000,122958.260
LAMI.RD308.D9.C3,3076.9688,687527.06,281896.700,262886.470,0.0000,0.000,65659.700
LAMI.RD319.D2.C2,0.0000,538412.94,255542.720,121803.890,0.0000,0.000,73231.560
LAMI.RD319.D2.C3,1407.5230,106901.95,59064.336,23485.404,0.0000,0.000,55986.290


In [46]:
# Convert to relative abundance (row-wise normalization)
metaB_V1V3_tbl_relative = metaB_V1V3_tbl.div(metaB_V1V3_tbl.sum(axis=1), axis=0)
metaB_V1V3_tbl_relative

,Urocanic acid,Isoleucine,Phenylalanine,Tryptophan,N-acyl lipid glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,0.009456,0.480551,0.230237,0.134635,0.000000,0.000000,0.145121
LAMI.RD304.D11.C1,0.014256,0.556292,0.231034,0.127297,0.002085,0.038587,0.030449
LAMI.RD304.D11.C2,0.000000,0.445655,0.191746,0.096169,0.001698,0.099395,0.165337
LAMI.RD304.D0.C1,0.002036,0.473830,0.178553,0.083500,0.002088,0.000000,0.259992
LAMI.RD304.D0.C2,0.005781,0.575333,0.243249,0.106394,0.003171,0.014404,0.051668
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.003977,0.508019,0.235307,0.202731,0.000000,0.000000,0.049966
LAMI.RD308.D9.C3,0.002365,0.528441,0.216669,0.202058,0.000000,0.000000,0.050467
LAMI.RD319.D2.C2,0.000000,0.544406,0.258387,0.123160,0.000000,0.000000,0.074047
LAMI.RD319.D2.C3,0.005702,0.433072,0.239277,0.095142,0.000000,0.000000,0.226807


In [47]:
# Filter metadata for the correct group
filtered_metadata = metadata[metadata['group'] == skin_group]

# Normalize SampleID and index formatting
metaB_V1V3_tbl_relative.index = metaB_V1V3_tbl_relative.index.str.strip().str.upper()
filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()

# Filter metadata to only include matching SampleIDs
filtered_metadata = filtered_metadata[filtered_metadata['SampleID'].isin(metaB_V1V3_tbl_relative.index)]

# Subset the DataFrame
metaB_V1V3_tbl_relative = metaB_V1V3_tbl_relative.loc[filtered_metadata['SampleID']]

metaB_V1V3_tbl_relative

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_2738/3342871550.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()


,Urocanic acid,Isoleucine,Phenylalanine,Tryptophan,N-acyl lipid glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD310.D21.C1,0.097339,0.528035,0.231706,0.131145,0.000000,0.000000,0.011775
LAMI.RD306.D18.C2,0.007350,0.534449,0.238542,0.128946,0.001350,0.026710,0.062652
LAMI.RD317.D14.C2,0.003761,0.485158,0.218103,0.125652,0.003487,0.001620,0.162218
LAMI.RD305.D23.C1,0.000000,0.405961,0.154526,0.085239,0.000000,0.000000,0.354274
LAMI.RD304.D7.C1,0.013503,0.569706,0.260374,0.125739,0.003758,0.013751,0.013170
...,...,...,...,...,...,...,...
LAMI.RD305.D4.C1,0.000000,0.534474,0.239424,0.128785,0.000000,0.000000,0.097317
LAMI.RD305.D28.C1,0.000000,0.452663,0.167551,0.091357,0.000000,0.000000,0.288429
LAMI.RD313.D4.C1,0.006726,0.553156,0.265829,0.133946,0.002669,0.000000,0.037674
LAMI.RD305.D0.C2,0.015392,0.467858,0.215846,0.115826,0.000000,0.000000,0.185078


In [48]:
# Save as biom for mmvec
metaB_V1V3_tbl_transposed = metaB_V1V3_tbl_relative.T
output_path = f'../Data/multi-omics/metabolomics-tbl_16S_V1V3-matched_relative_{skin_group}.biom'
save_as_biom(metaB_V1V3_tbl_transposed, output_path)

In [49]:
# Subset V1V3_tbl rows where sample is in metaB_tbl
V1V3_tbl_matched = V1V3_tbl[V1V3_tbl.index.isin(metaB_V1V3_tbl_relative.index)]

# top_V1V3_features = [' g__Cutibacterium', ' g__Staphylococcus',
#                      ' g__Streptococcus', ' g__Corynebacterium', ' g__Lawsonella',
#                      ' g__Micrococcus', ' g__Alloprevotella', ' g__Lactobacillus', 
#                      ' g__Rothia', ' g__Kocuria', ' g__Haemophilus']

top_V1V3_features = [' g__Cutibacterium', ' g__Staphylococcus',
                     ' g__Streptococcus',' g__Lawsonella',
                     ' g__Micrococcus', ' g__Lactobacillus', 
                     ' g__Rothia', ' g__Kocuria', ' g__Haemophilus']

# Find the intersection of desired columns and actual DataFrame columns
available_columns = V1V3_tbl_matched.columns.intersection(top_V1V3_features)
V1V3_tbl_matched = V1V3_tbl_matched[available_columns]

V1V3_tbl_matched

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Lactobacillus,g__Rothia,g__Haemophilus,g__Kocuria
LAMI.RD305.D14.C1,2900.0,30.0,5.0,7.0,0.0,0.0,3.0,0.0,0.0
LAMI.RD304.D4.C2,3105.0,535.0,9.0,3.0,8.0,64.0,0.0,0.0,0.0
LAMI.RD304.D16.C1,3416.0,122.0,15.0,8.0,1.0,91.0,0.0,0.0,0.0
LAMI.RD304.D7.C1,3377.0,228.0,15.0,14.0,2.0,57.0,0.0,1.0,1.0
LAMI.RD304.D18.C2,3574.0,150.0,2.0,4.0,8.0,3.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D21.C1,2010.0,6.0,1.0,19.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D21.C2,2611.0,10.0,0.0,15.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C1,1900.0,8.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2315.0,8.0,0.0,27.0,0.0,0.0,0.0,0.0,0.0


In [50]:
# Convert to relative abundance (row-wise normalization)
V1V3_tbl_matched_relative = V1V3_tbl_matched.div(V1V3_tbl_matched.sum(axis=1), axis=0)
V1V3_tbl_matched_relative

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Lawsonella,g__Micrococcus,g__Lactobacillus,g__Rothia,g__Haemophilus,g__Kocuria
LAMI.RD305.D14.C1,0.984720,0.010187,0.001698,0.002377,0.000000,0.000000,0.001019,0.000000,0.000000
LAMI.RD304.D4.C2,0.833781,0.143663,0.002417,0.000806,0.002148,0.017186,0.000000,0.000000,0.000000
LAMI.RD304.D16.C1,0.935122,0.033397,0.004106,0.002190,0.000274,0.024911,0.000000,0.000000,0.000000
LAMI.RD304.D7.C1,0.913938,0.061705,0.004060,0.003789,0.000541,0.015426,0.000000,0.000271,0.000271
LAMI.RD304.D18.C2,0.955360,0.040096,0.000535,0.001069,0.002138,0.000802,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D21.C1,0.987230,0.002947,0.000491,0.009332,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD319.D21.C2,0.990516,0.003794,0.000000,0.005690,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD319.D14.C1,0.991649,0.004175,0.000000,0.004175,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD319.D14.C2,0.985106,0.003404,0.000000,0.011489,0.000000,0.000000,0.000000,0.000000,0.000000


In [51]:
# Save metadata for V1V3
metadata_V1V3 = metadata[metadata['SampleID'].isin(V1V3_tbl_matched_relative.index)]
metadata_V1V3 = metadata_V1V3[['SampleID', 'group']]

# Rename the first column to 'sample id'
metadata_V1V3.rename(columns={metadata_V1V3.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V1V3.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V1V3.columns[1:])]

metadata_V1V3.to_csv('../Data/multi-omics/16S_V1V3_metaB-matched.csv', index=0)
metadata_V1V3

,sample id,group
1,LAMI.RD310.D21.C1,Acne_L
3,LAMI.RD306.D18.C2,Acne_L
6,LAMI.RD317.D14.C2,Acne_L
7,LAMI.RD305.D23.C1,Acne_L
8,LAMI.RD304.D7.C1,Acne_L
...,...,...
257,LAMI.RD305.D4.C1,Acne_L
258,LAMI.RD305.D28.C1,Acne_L
259,LAMI.RD313.D4.C1,Acne_L
260,LAMI.RD305.D0.C2,Acne_L


In [52]:
# Save as biom for mmvec
V1V3_tbl_matched_transposed = V1V3_tbl_matched_relative.T
output_path = f'../Data/multi-omics/16S_V1V3-tbl_metaB-matched_relative_{skin_group}.biom'
save_as_biom(V1V3_tbl_matched_transposed, output_path)

In [53]:
# Create column_names_df with 'feature id' as the column header
column_names_df = pd.DataFrame({"feature id": metaB_V1V3_tbl.columns})  # Ensure feature id is a string

column_names_df

,feature id
0,Urocanic acid
1,Isoleucine
2,Phenylalanine
3,Tryptophan
4,N-acyl lipid glutamine-C14:0
5,Sorbitane Monooleate
6,C19H36O4 Fatty alcohol


In [54]:
# Save to a CSV file
output_path = "../Data/multi-omics/metabolite_info.csv"
column_names_df = column_names_df.to_csv(output_path, index=0)

In [55]:
# Subset metaB rows where sample is in V4_tbl
metaB_V4_tbl = metaB_tbl[metaB_tbl.index.isin(V4_tbl.index)]
# metaB_V4_tbl = metaB_V4_tbl.drop('group', axis=1)
metaB_V4_tbl.index.name = None
metaB_V4_tbl

,Urocanic acid,Isoleucine,Phenylalanine,Tryptophan,N-acyl lipid glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,39859.7580,2025607.40,970487.500,567510.060,0.0000,0.000,611709.700
LAMI.RD304.D11.C1,36750.4000,1434033.20,595568.940,328151.120,5375.8525,99470.690,78491.530
LAMI.RD304.D11.C2,0.0000,706197.75,303845.300,152392.720,2691.0256,157503.920,261996.800
LAMI.RD304.D0.C1,3345.8184,778566.10,293386.340,137201.390,3431.0034,0.000,427202.060
LAMI.RD304.D0.C2,24984.3220,2486485.20,1051277.400,459814.620,13703.5760,62253.426,223301.700
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,1250164.60,579058.800,498893.780,0.0000,0.000,122958.260
LAMI.RD308.D9.C3,3076.9688,687527.06,281896.700,262886.470,0.0000,0.000,65659.700
LAMI.RD319.D2.C2,0.0000,538412.94,255542.720,121803.890,0.0000,0.000,73231.560
LAMI.RD319.D2.C3,1407.5230,106901.95,59064.336,23485.404,0.0000,0.000,55986.290


In [56]:
# Drop the column from the DataFrame as it wasn't significant after univariate testing
# metaB_V4_tbl = metaB_V4_tbl.drop(columns=['C19H22O4 Linear diarylheptanoids'])

In [57]:
# Convert to relative abundance (row-wise normalization)
metaB_V4_tbl_relative = metaB_V4_tbl.div(metaB_V4_tbl.sum(axis=1), axis=0)
metaB_V4_tbl_relative

,Urocanic acid,Isoleucine,Phenylalanine,Tryptophan,N-acyl lipid glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,0.009456,0.480551,0.230237,0.134635,0.000000,0.000000,0.145121
LAMI.RD304.D11.C1,0.014256,0.556292,0.231034,0.127297,0.002085,0.038587,0.030449
LAMI.RD304.D11.C2,0.000000,0.445655,0.191746,0.096169,0.001698,0.099395,0.165337
LAMI.RD304.D0.C1,0.002036,0.473830,0.178553,0.083500,0.002088,0.000000,0.259992
LAMI.RD304.D0.C2,0.005781,0.575333,0.243249,0.106394,0.003171,0.014404,0.051668
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.003977,0.508019,0.235307,0.202731,0.000000,0.000000,0.049966
LAMI.RD308.D9.C3,0.002365,0.528441,0.216669,0.202058,0.000000,0.000000,0.050467
LAMI.RD319.D2.C2,0.000000,0.544406,0.258387,0.123160,0.000000,0.000000,0.074047
LAMI.RD319.D2.C3,0.005702,0.433072,0.239277,0.095142,0.000000,0.000000,0.226807


In [58]:
# Filter metadata for the correct group
filtered_metadata = metadata[metadata['group'] == skin_group]

# Normalize SampleID and index formatting
metaB_V4_tbl_relative.index = metaB_V4_tbl_relative.index.str.strip().str.upper()
filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()

# Filter metadata to only include matching SampleIDs
filtered_metadata = filtered_metadata[filtered_metadata['SampleID'].isin(metaB_V4_tbl_relative.index)]

# Subset the DataFrame
metaB_V4_tbl_relative = metaB_V4_tbl_relative.loc[filtered_metadata['SampleID']]

metaB_V4_tbl_relative

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_2738/4148729606.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_metadata['SampleID'] = filtered_metadata['SampleID'].str.strip().str.upper()


,Urocanic acid,Isoleucine,Phenylalanine,Tryptophan,N-acyl lipid glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD310.D21.C1,0.097339,0.528035,0.231706,0.131145,0.000000,0.000000,0.011775
LAMI.RD306.D18.C2,0.007350,0.534449,0.238542,0.128946,0.001350,0.026710,0.062652
LAMI.RD317.D14.C2,0.003761,0.485158,0.218103,0.125652,0.003487,0.001620,0.162218
LAMI.RD304.D7.C1,0.013503,0.569706,0.260374,0.125739,0.003758,0.013751,0.013170
LAMI.RD305.D21.C2,0.000000,0.461077,0.200479,0.090830,0.000000,0.000000,0.247613
...,...,...,...,...,...,...,...
LAMI.RD305.D4.C1,0.000000,0.534474,0.239424,0.128785,0.000000,0.000000,0.097317
LAMI.RD305.D28.C1,0.000000,0.452663,0.167551,0.091357,0.000000,0.000000,0.288429
LAMI.RD313.D4.C1,0.006726,0.553156,0.265829,0.133946,0.002669,0.000000,0.037674
LAMI.RD305.D0.C2,0.015392,0.467858,0.215846,0.115826,0.000000,0.000000,0.185078


In [59]:
# Subset V4_tbl rows where sample is in metaB_tbl
V4_tbl_matched = V4_tbl[V4_tbl.index.isin(metaB_V4_tbl_relative.index)]

# top_V4_features = [' g__Staphylococcus', ' g__Lawsonella',
#                     ' g__Streptococcus', ' g__Acinetobacter', ' g__Cutibacterium', 'g__Enhydrobacter',
#                      ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia', ' g__Kocuria', ' g__Haemophilus', ' g__Corynebacterium']


top_V4_features = [' g__Staphylococcus', ' g__Lawsonella', ' g__Streptococcus', ' g__Cutibacterium',
                     ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia', ' g__Kocuria', ' g__Haemophilus']

# Find the intersection of desired columns and actual DataFrame columns
available_columns = V4_tbl_matched.columns.intersection(top_V4_features)
V4_tbl_matched = V4_tbl_matched[available_columns]

V4_tbl_matched

,g__Staphylococcus,g__Lawsonella,g__Streptococcus,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Cutibacterium,g__Kocuria,g__Rothia
LAMI.RD304.D0.C1,2249.0,22.0,105.0,9.0,26.0,65.0,16.0,5.0,3.0
LAMI.RD304.D0.C2,2410.0,57.0,102.0,17.0,20.0,91.0,12.0,3.0,2.0
LAMI.RD304.D18.C2,2409.0,72.0,37.0,75.0,90.0,37.0,42.0,11.0,2.0
LAMI.RD304.D9.C1,2977.0,48.0,24.0,5.0,21.0,136.0,13.0,9.0,0.0
LAMI.RD304.D21.C2,1478.0,45.0,299.0,45.0,51.0,456.0,29.0,3.0,6.0
...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D25.C2,68.0,54.0,1.0,0.0,4.0,0.0,1.0,0.0,0.0
LAMI.RD319.D28.C1,42.0,96.0,7.0,9.0,0.0,0.0,0.0,0.0,2.0
LAMI.RD318.D9.C2,139.0,23.0,11.0,2.0,12.0,2.0,10.0,0.0,1.0
LAMI.RD319.D28.C2,52.0,54.0,1.0,1.0,2.0,1.0,0.0,0.0,0.0


In [60]:
# Convert to relative abundance (row-wise normalization)
V4_tbl_matched_relative = V4_tbl_matched.div(V4_tbl_matched.sum(axis=1), axis=0)
V4_tbl_matched_relative

,g__Staphylococcus,g__Lawsonella,g__Streptococcus,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Cutibacterium,g__Kocuria,g__Rothia
LAMI.RD304.D0.C1,0.899600,0.008800,0.042000,0.003600,0.010400,0.026000,0.006400,0.002000,0.001200
LAMI.RD304.D0.C2,0.887988,0.021002,0.037583,0.006264,0.007369,0.033530,0.004422,0.001105,0.000737
LAMI.RD304.D18.C2,0.868108,0.025946,0.013333,0.027027,0.032432,0.013333,0.015135,0.003964,0.000721
LAMI.RD304.D9.C1,0.920817,0.014847,0.007423,0.001547,0.006496,0.042066,0.004021,0.002784,0.000000
LAMI.RD304.D21.C2,0.612769,0.018657,0.123964,0.018657,0.021144,0.189055,0.012023,0.001244,0.002488
...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D25.C2,0.531250,0.421875,0.007812,0.000000,0.031250,0.000000,0.007812,0.000000,0.000000
LAMI.RD319.D28.C1,0.269231,0.615385,0.044872,0.057692,0.000000,0.000000,0.000000,0.000000,0.012821
LAMI.RD318.D9.C2,0.695000,0.115000,0.055000,0.010000,0.060000,0.010000,0.050000,0.000000,0.005000
LAMI.RD319.D28.C2,0.468468,0.486486,0.009009,0.009009,0.018018,0.009009,0.000000,0.000000,0.000000


In [61]:
# Save metadata for V4
metadata_V4 = metadata[metadata['SampleID'].isin(V4_tbl_matched.index)]
metadata_V4 = metadata_V4[['SampleID', 'group']]
# Rename the first column to 'sample id'
metadata_V4.rename(columns={metadata_V4.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V4.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V4.columns[1:])]

metadata_V4.to_csv('../Data/multi-omics/16S_V4_metaB-matched.csv', index=0)
metadata_V4

,sample id,group
1,LAMI.RD310.D21.C1,Acne_L
3,LAMI.RD306.D18.C2,Acne_L
6,LAMI.RD317.D14.C2,Acne_L
8,LAMI.RD304.D7.C1,Acne_L
10,LAMI.RD305.D21.C2,Acne_L
...,...,...
257,LAMI.RD305.D4.C1,Acne_L
258,LAMI.RD305.D28.C1,Acne_L
259,LAMI.RD313.D4.C1,Acne_L
260,LAMI.RD305.D0.C2,Acne_L


In [62]:
# Save as biom for mmvec
metaB_V4_tbl_transposed = metaB_V4_tbl_relative.T
output_path = f'../Data/multi-omics/metabolomics-tbl_16S_V4-matched_relative_{skin_group}.biom'
save_as_biom(metaB_V4_tbl_transposed, output_path)

In [63]:
# Save as biom for mmvec
V4_tbl_matched_transposed = V4_tbl_matched_relative.T
output_path = '../Data/multi-omics/16S_V4-tbl_metaB-matched_relative.biom'
save_as_biom(V4_tbl_matched_transposed, output_path)

### Combine V1-V3 and V4 tables

In [64]:
# # Find common rows (samples)
# common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# # Subset both DataFrames to keep only common rows
# V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
# V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# # Merge by taking average of values row-wise
# merged_tbl = V1V3_tbl_matched.add(V4_tbl_matched, fill_value=0)

# # Sort columns by total sum (descending order)
# merged_tbl = merged_tbl[merged_tbl.sum().sort_values(ascending=False).index]

# merged_tbl

In [65]:
V1V3_tbl_matched.columns

Index([' g__Cutibacterium', ' g__Staphylococcus', ' g__Streptococcus',
       ' g__Lawsonella', ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia',
       ' g__Haemophilus', ' g__Kocuria'],
      dtype='object')

In [66]:
V4_tbl_matched.columns

Index([' g__Staphylococcus', ' g__Lawsonella', ' g__Streptococcus',
       ' g__Haemophilus', ' g__Micrococcus', ' g__Lactobacillus',
       ' g__Cutibacterium', ' g__Kocuria', ' g__Rothia'],
      dtype='object')

In [67]:
# Find common rows (samples)
common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# Subset both DataFrames to keep only common rows
V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# Compute the mean for each matching row and column
merged_tbl = (V1V3_tbl_matched + V4_tbl_matched) / 2

# Sort columns by total mean (descending order)
merged_tbl = merged_tbl[merged_tbl.mean().sort_values(ascending=False).index]

# Display merged DataFrame
merged_tbl


,g__Cutibacterium,g__Staphylococcus,g__Lawsonella,g__Streptococcus,g__Lactobacillus,g__Haemophilus,g__Micrococcus,g__Kocuria,g__Rothia
LAMI.RD305.D14.C1,1452.5,128.0,30.0,10.0,0.0,7.0,3.5,0.0,1.5
LAMI.RD304.D4.C2,1560.0,1743.0,10.5,19.5,143.0,3.5,21.5,4.5,0.0
LAMI.RD304.D16.C1,1717.5,791.0,47.0,54.5,366.5,17.0,10.0,3.5,3.0
LAMI.RD304.D7.C1,1694.0,1233.0,58.5,41.0,196.0,10.5,27.5,4.0,2.0
LAMI.RD304.D18.C2,1808.0,1279.5,38.0,19.5,20.0,37.5,49.0,5.5,1.0
...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D21.C1,1005.5,24.0,19.5,1.5,0.5,0.0,1.5,0.0,0.0
LAMI.RD319.D21.C2,1306.0,33.5,19.0,0.0,0.0,0.0,0.5,0.0,0.0
LAMI.RD319.D14.C1,950.5,18.0,7.0,0.5,0.5,0.0,1.0,0.0,0.0
LAMI.RD319.D14.C2,1157.5,40.0,30.0,1.5,0.5,0.5,0.5,0.0,0.0


In [68]:
# Convert to relative abundance (row-wise normalization)
merged_tbl_relative = merged_tbl.div(merged_tbl.sum(axis=1), axis=0)
merged_tbl_relative

,g__Cutibacterium,g__Staphylococcus,g__Lawsonella,g__Streptococcus,g__Lactobacillus,g__Haemophilus,g__Micrococcus,g__Kocuria,g__Rothia
LAMI.RD305.D14.C1,0.889740,0.078407,0.018377,0.006126,0.000000,0.004288,0.002144,0.000000,0.000919
LAMI.RD304.D4.C2,0.445015,0.497219,0.002995,0.005563,0.040793,0.000998,0.006133,0.001284,0.000000
LAMI.RD304.D16.C1,0.570598,0.262791,0.015615,0.018106,0.121761,0.005648,0.003322,0.001163,0.000997
LAMI.RD304.D7.C1,0.518598,0.377468,0.017909,0.012552,0.060003,0.003214,0.008419,0.001225,0.000612
LAMI.RD304.D18.C2,0.554942,0.392726,0.011664,0.005985,0.006139,0.011510,0.015040,0.001688,0.000307
...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D21.C1,0.955344,0.022803,0.018527,0.001425,0.000475,0.000000,0.001425,0.000000,0.000000
LAMI.RD319.D21.C2,0.961001,0.024650,0.013981,0.000000,0.000000,0.000000,0.000368,0.000000,0.000000
LAMI.RD319.D14.C1,0.972379,0.018414,0.007161,0.000512,0.000512,0.000000,0.001023,0.000000,0.000000
LAMI.RD319.D14.C2,0.940675,0.032507,0.024380,0.001219,0.000406,0.000406,0.000406,0.000000,0.000000


In [69]:
merged_tbl_relative.columns

Index([' g__Cutibacterium', ' g__Staphylococcus', ' g__Lawsonella',
       ' g__Streptococcus', ' g__Lactobacillus', ' g__Haemophilus',
       ' g__Micrococcus', ' g__Kocuria', ' g__Rothia'],
      dtype='object')

In [70]:
# Save as biom for mmvec
merged_tbl_relative_transposed = merged_tbl_relative.T
output_path = f'../Data/multi-omics/16S_MERGED-Avg-tbl_metaB-matched_relative_{skin_group}.biom'
save_as_biom(merged_tbl_relative_transposed, output_path)

In [71]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_MERGED_tbl = metaB_tbl[metaB_tbl.index.isin(merged_tbl_relative.index)]

# Remove the 'group' column (only for VIP table)
# metaB_MERGED_tbl = metaB_MERGED_tbl.drop(columns=['group'])
metaB_MERGED_tbl.index.name = None

In [72]:
# Sort columns by total sum (descending order)
metaB_MERGED_tbl = metaB_MERGED_tbl[metaB_MERGED_tbl.sum().sort_values(ascending=False).index]
metaB_MERGED_tbl

,Isoleucine,Phenylalanine,C19H36O4 Fatty alcohol,Tryptophan,Sorbitane Monooleate,Urocanic acid,N-acyl lipid glutamine-C14:0
LAMI.RD304.D11.C1,1434033.20,595568.94,78491.530,328151.12,99470.690,36750.4000,5375.8525
LAMI.RD304.D11.C2,706197.75,303845.30,261996.800,152392.72,157503.920,0.0000,2691.0256
LAMI.RD304.D0.C1,778566.10,293386.34,427202.060,137201.39,0.000,3345.8184,3431.0034
LAMI.RD304.D0.C2,2486485.20,1051277.40,223301.700,459814.62,62253.426,24984.3220,13703.5760
LAMI.RD305.D0.C1,792388.94,330585.60,396271.840,239201.12,0.000,2908.8442,0.0000
...,...,...,...,...,...,...,...
LAMI.RD310.D0.C2,1394403.50,569740.44,502334.800,376634.80,0.000,7607.8213,0.0000
LAMI.RD304.D9.C1,908266.90,457527.78,1562542.800,219415.83,0.000,0.0000,9194.5180
LAMI.RD304.D9.C2,1116758.50,521042.84,251054.770,267440.38,0.000,0.0000,12874.5510
LAMI.RD319.D2.C2,538412.94,255542.72,73231.560,121803.89,0.000,0.0000,0.0000


In [74]:
# Convert to relative abundance (row-wise normalization)
metaB_MERGED_tbl_relative = metaB_MERGED_tbl.div(metaB_MERGED_tbl.sum(axis=1), axis=0)
metaB_MERGED_tbl_relative

,Isoleucine,Phenylalanine,C19H36O4 Fatty alcohol,Tryptophan,Sorbitane Monooleate,Urocanic acid,N-acyl lipid glutamine-C14:0
LAMI.RD304.D11.C1,0.556292,0.231034,0.030449,0.127297,0.038587,0.014256,0.002085
LAMI.RD304.D11.C2,0.445655,0.191746,0.165337,0.096169,0.099395,0.000000,0.001698
LAMI.RD304.D0.C1,0.473830,0.178553,0.259992,0.083500,0.000000,0.002036,0.002088
LAMI.RD304.D0.C2,0.575333,0.243249,0.051668,0.106394,0.014404,0.005781,0.003171
LAMI.RD305.D0.C1,0.449874,0.187688,0.224981,0.135805,0.000000,0.001651,0.000000
...,...,...,...,...,...,...,...
LAMI.RD310.D0.C2,0.489141,0.199858,0.176213,0.132119,0.000000,0.002669,0.000000
LAMI.RD304.D9.C1,0.287704,0.144927,0.494954,0.069503,0.000000,0.000000,0.002912
LAMI.RD304.D9.C2,0.514832,0.240204,0.115738,0.123292,0.000000,0.000000,0.005935
LAMI.RD319.D2.C2,0.544406,0.258387,0.074047,0.123160,0.000000,0.000000,0.000000


In [75]:
# Save as biom for mmvec
metaB_MERGED_tbl_relative_transposed = metaB_MERGED_tbl_relative.T
output_path = f'../Data/multi-omics/metaB_MERGED-Avg-tbl_matched_relative_{skin_group}.biom'
save_as_biom(metaB_MERGED_tbl_relative_transposed, output_path)

In [76]:
metaB_MERGED_tbl_relative.columns

Index(['Isoleucine', 'Phenylalanine', 'C19H36O4 Fatty alcohol', 'Tryptophan',
       'Sorbitane Monooleate', 'Urocanic acid',
       'N-acyl lipid glutamine-C14:0'],
      dtype='object')